# Pipeline tầng tri thức và tầng sinh nội dung — Báo cáo

Demo trên dữ liệu công khai Wikidata (lịch sử Việt Nam), theo yêu cầu của
giáo viên hướng dẫn: chứng minh pipeline chạy được và hiểu rõ từng bước,
trước khi áp dụng lên dữ liệu lịch sử Việt Nam tự soạn (giai đoạn 1858–1945)
ở giai đoạn sau.

## Bước 1 — Trích subgraph lịch sử Việt Nam từ Wikidata

Wikidata là một cơ sở tri thức mở: mỗi thực thể có một mã định danh (QID,
ví dụ Hồ Chí Minh là `Q10520`), mỗi loại quan hệ có một mã thuộc tính (PID,
ví dụ "quốc gia" là `P17`). Bước này dùng SPARQL — ngôn ngữ truy vấn chuẩn
cho dữ liệu RDF/triple — để lấy về các thực thể liên quan lịch sử Việt Nam
cùng quan hệ giữa chúng, dưới dạng triples `(head, relation, tail)`.

Truy vấn lấy mọi thực thể có `wdt:P17` (quốc gia) là Việt Nam (`wd:Q881`),
cùng các quan hệ: người tham gia (`P710`), thời điểm (`P585`), thời điểm bắt
đầu/kết thúc (`P580`/`P582`), địa điểm (`P276`), nguyên nhân (`P828`), vị trí
/chức vụ (`P39`). Độ phủ thật của Wikidata cho từng quan hệ chưa biết trước
— số liệu thật được ghi lại ở phần "Chạy thật" dưới đây, quyết định có cần
mở rộng phạm vi lọc hay không dựa trên số liệu đó, không đoán trước.

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import csv

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
VIETNAM_QID = "Q881"


def build_sparql_query(country_qid: str = VIETNAM_QID, limit: int = 1000) -> str:
    """Trả về câu SPARQL lấy thực thể liên quan lịch sử Việt Nam kèm quan hệ."""
    return f"""
    SELECT ?head ?headLabel ?relation ?relationLabel ?tail ?tailLabel WHERE {{
      ?head wdt:P17 wd:{country_qid} .
      {{ ?head wdt:P710 ?tail . BIND(wd:P710 AS ?relation) }}
      UNION {{ ?head wdt:P585 ?tail . BIND(wd:P585 AS ?relation) }}
      UNION {{ ?head wdt:P580 ?tail . BIND(wd:P580 AS ?relation) }}
      UNION {{ ?head wdt:P582 ?tail . BIND(wd:P582 AS ?relation) }}
      UNION {{ ?head wdt:P276 ?tail . BIND(wd:P276 AS ?relation) }}
      UNION {{ ?head wdt:P828 ?tail . BIND(wd:P828 AS ?relation) }}
      UNION {{ ?head wdt:P39 ?tail . BIND(wd:P39 AS ?relation) }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "vi,en". }}
    }}
    LIMIT {limit}
    """


def run_query(query: str, endpoint: str = WIKIDATA_ENDPOINT) -> list:
    """Gọi SPARQL endpoint, trả về list các dict bindings thô."""
    sparql = SPARQLWrapper(endpoint, agent="hddhistory-kg-research/0.1")
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    return results["results"]["bindings"]


def _qid_from_uri(uri: str) -> str:
    """Lấy QID/PID từ URI Wikidata, ví dụ '.../entity/Q881' -> 'Q881'."""
    return uri.rsplit("/", 1)[-1]


def _tail_value(binding_tail: dict) -> str:
    """Nếu tail là một thực thể (URI) thì lấy QID, nếu là literal (ví dụ ngày
    tháng) thì giữ nguyên giá trị."""
    if binding_tail.get("type") == "uri":
        return _qid_from_uri(binding_tail["value"])
    return binding_tail["value"]


def bindings_to_triples(bindings: list) -> list:
    """Chuyển kết quả SPARQL thô thành list triples có cấu trúc."""
    triples = []
    for b in bindings:
        triples.append({
            "head_qid": _qid_from_uri(b["head"]["value"]),
            "head_label": b.get("headLabel", {}).get("value", ""),
            "relation_pid": _qid_from_uri(b["relation"]["value"]),
            "relation_label": b.get("relationLabel", {}).get("value", ""),
            "tail_qid": _tail_value(b["tail"]),
            "tail_label": b.get("tailLabel", {}).get("value", ""),
        })
    return triples


def save_triples_csv(triples: list, path: str) -> None:
    """Lưu triples ra CSV."""
    fieldnames = ["head_qid", "head_label", "relation_pid", "relation_label",
                  "tail_qid", "tail_label"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(triples)

### Ví dụ minh hoạ (chạy thật)

Truy vấn với limit=1000 lấy được **1000 triples**. Kết quả đạt đủ 1000 triples
(chính xác là limit đã đặt), điều này cho thấy còn dữ liệu ngoài giới hạn trong
Wikidata. Phạm vi hiện tại (7 loại quan hệ + điều kiện quốc gia Việt Nam) đã
đủ cho Task 1, và dữ liệu được lưu vào `data/raw/triples.csv`.

In [ ]:
if __name__ == "__main__":
    import os
    query = build_sparql_query(limit=1000)
    bindings = run_query(query)
    triples = bindings_to_triples(bindings)
    print(f"Số triples lấy được: {len(triples)}")
    os.makedirs("data/raw", exist_ok=True)
    save_triples_csv(triples, "data/raw/triples.csv")

## Bước 2 — Nạp subgraph vào Neo4j

Neo4j là một cơ sở dữ liệu đồ thị: dữ liệu được lưu trực tiếp dưới dạng nút
(node) và cạnh (relationship) có thuộc tính, khác với việc mô phỏng đồ thị
bằng bảng quan hệ. Bước này đọc file triples từ Bước 1 và nạp vào Neo4j
bằng lệnh Cypher `MERGE` — `MERGE` chỉ tạo mới nếu nút/cạnh chưa tồn tại,
nên chạy lại script nhiều lần không tạo dữ liệu trùng.

Mỗi thực thể trở thành một nút nhãn `Entity` với thuộc tính `qid` và
`label`. Mọi quan hệ dùng chung một loại cạnh Cypher `RELATION`, mang thuộc
tính `pid` và `label` riêng — thay vì tạo một loại cạnh khác nhau cho mỗi
quan hệ — để Bước 5 (truy vấn multi-hop) không phải liệt kê trước tên loại
quan hệ cụ thể.

In [ ]:
import csv
from neo4j import GraphDatabase


def get_driver(uri: str, user: str, password: str):
    """Tạo Neo4j driver."""
    return GraphDatabase.driver(uri, auth=(user, password))


def _load_one_triple(tx, triple: dict) -> None:
    tx.run(
        """
        MERGE (h:Entity {qid: $head_qid})
        SET h.label = $head_label
        MERGE (t:Entity {qid: $tail_qid})
        SET t.label = $tail_label
        MERGE (h)-[r:RELATION {pid: $relation_pid}]->(t)
        SET r.label = $relation_label
        """,
        head_qid=triple["head_qid"],
        head_label=triple["head_label"],
        tail_qid=triple["tail_qid"],
        tail_label=triple["tail_label"],
        relation_pid=triple["relation_pid"],
        relation_label=triple["relation_label"],
    )


def load_triples(driver, triples: list) -> dict:
    """Nạp triples vào Neo4j, trả về {"triples_loaded": int}."""
    with driver.session() as session:
        for triple in triples:
            session.execute_write(_load_one_triple, triple)
    return {"triples_loaded": len(triples)}


def load_triples_from_csv(driver, csv_path: str) -> dict:
    """Đọc CSV rồi gọi load_triples."""
    with open(csv_path, encoding="utf-8") as f:
        triples = list(csv.DictReader(f))
    return load_triples(driver, triples)


def count_nodes_and_relationships(driver) -> dict:
    """Đếm số nút/cạnh thật đang có trong Neo4j — đối chiếu với báo cáo."""
    with driver.session() as session:
        node_count = session.run("MATCH (n:Entity) RETURN count(n) AS c").single()["c"]
        rel_count = session.run("MATCH ()-[r:RELATION]->() RETURN count(r) AS c").single()["c"]
    return {"node_count": node_count, "relationship_count": rel_count}

### Ví dụ minh hoạ (chạy thật)

Chạy `python -m src.step2_load_neo4j` để nạp toàn bộ `data/raw/triples.csv`
(1000 dòng, sinh ra ở Bước 1) vào container Neo4j khởi động bằng
`docker compose up -d`. Kết quả in ra từ lần chạy thật trên máy này:

```
Đã nạp 1000 triples
Neo4j hiện có 1017 nút, 1000 cạnh
```

- Số triples nạp: **1000**
- Số nút (`Entity`) sau khi nạp: **1017**
- Số cạnh (`RELATION`) sau khi nạp: **1000**

Số nút (1017) nhỏ hơn nhiều so với 2 × số triples (2000) vì phần lớn thực
thể (ví dụ các nhân vật, địa danh, sự kiện lịch sử Việt Nam xuất hiện lặp
lại) là head hoặc tail của nhiều triples khác nhau — `MERGE (h:Entity
{qid: ...})` chỉ tạo nút mới lần đầu gặp một `qid`, các lần sau chỉ cập
nhật thuộc tính `label` trên nút đã có. Đã kiểm chứng tính idempotent của
`MERGE`: chạy lại `python -m src.step2_load_neo4j` lần hai cho đúng số liệu
trên (1017 nút, 1000 cạnh) — không tạo dữ liệu trùng.

In [ ]:
if __name__ == "__main__":
    driver = get_driver("bolt://localhost:7687", "neo4j", "kg-research-pw")
    result = load_triples_from_csv(driver, "data/raw/triples.csv")
    counts = count_nodes_and_relationships(driver)
    print(f"Đã nạp {result['triples_loaded']} triples")
    print(f"Neo4j hiện có {counts['node_count']} nút, {counts['relationship_count']} cạnh")
    driver.close()

## Bước 3 — Huấn luyện Knowledge Graph Embeddings (KGE)

KGE học một vector số (embedding) cho mỗi thực thể và mỗi loại quan hệ, sao
cho cấu trúc đồ thị được giữ lại trong không gian vector. Bước này dùng
**TransE** — mô hình KGE cổ điển, dễ diễn giải nhất — làm baseline.

**Công thức điểm số TransE:** với một triple `(h, r, t)`, TransE giả định
`h + r ≈ t` nếu triple đúng, và định nghĩa hàm điểm số

`score(h, r, t) = -‖ h + r - t ‖`

(dùng khoảng cách L1 hoặc L2; lấy dấu âm để điểm số cao hơn nghĩa là đúng
hơn). Mô hình huấn luyện bằng **margin ranking loss**: với mỗi triple đúng
`(h, r, t)`, sinh một triple sai bằng cách thay `h` hoặc `t` bằng một thực
thể ngẫu nhiên khác (negative sampling), rồi tối ưu để
`score(h, r, t) > score(h', r, t') + margin`.

**Đánh giá:** với mỗi triple trong tập test, mô hình xếp hạng mọi thực thể
có thể là `t` theo điểm số, rồi xem thực thể đúng đứng hạng thứ mấy.
- `MRR` (Mean Reciprocal Rank) = trung bình của `1/hạng` trên toàn tập test.
- `Hits@k` = tỉ lệ các triple mà thực thể đúng nằm trong top-k.

In [ ]:
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline as pykeen_pipeline
import pandas as pd
import json


def triples_csv_to_factory(csv_path: str) -> TriplesFactory:
    """Đọc triples CSV, trả về TriplesFactory dùng label làm định danh."""
    df = pd.read_csv(csv_path)
    triples_array = df[["head_label", "relation_label", "tail_label"]].to_numpy()
    return TriplesFactory.from_labeled_triples(triples_array)


def train_transe(tf: TriplesFactory, train_ratio: float = 0.8,
                  num_epochs: int = 100, random_seed: int = 42):
    """Chia train/test, huấn luyện TransE, trả về PipelineResult."""
    training, testing = tf.split([train_ratio, 1 - train_ratio],
                                  random_state=random_seed)
    result = pykeen_pipeline(
        training=training,
        testing=testing,
        model="TransE",
        random_seed=random_seed,
        training_kwargs=dict(num_epochs=num_epochs),
    )
    return result


def extract_eval_metrics(result) -> dict:
    """Trích MRR/Hits@k bằng API get_metric (ổn định qua các bản PyKEEN)."""
    mr = result.metric_results
    return {
        "mrr": mr.get_metric("mrr"),
        "hits_at_1": mr.get_metric("hits_at_1"),
        "hits_at_3": mr.get_metric("hits_at_3"),
        "hits_at_10": mr.get_metric("hits_at_10"),
    }


def save_eval_json(metrics: dict, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

### Ví dụ minh hoạ (chạy thật)

Chạy `python -m src.step3_train_kge` để huấn luyện TransE thật trên toàn bộ
`data/raw/triples.csv` (1000 dòng từ Bước 1; `TriplesFactory` gộp các triple
trùng lặp còn lại **997 triple duy nhất**, trên **1009 thực thể** và **6**
loại quan hệ), 100 epoch, `train_ratio` mặc định 0.8 (~797 triple train /
~200 triple test), `random_seed=42`. Kết quả đánh giá thật trên tập test, in
ra từ lần chạy này trên máy này (đã lưu vào `reports/kge_eval.json`):

```json
{
  "mrr": 0.021817443892359734,
  "hits_at_1": 0.0,
  "hits_at_3": 0.0175,
  "hits_at_10": 0.0475
}
```

- `MRR`: **0.0218** — nghịch đảo của nó (1/MRR ≈ 45.8) cho một ước lượng
  thô về độ lớn thứ hạng trung bình của thực thể đúng, trên tổng số 1009
  thực thể của đồ thị.
- `Hits@1`: **0%**, `Hits@3`: **1.75%**, `Hits@10`: **4.75%**.

Các số liệu này thấp — điều này hợp lý, không phải lỗi: 1000 triple được
trích ra từ Wikidata theo 7 loại quan hệ cố định (Bước 1) tạo thành một đồ
thị **rất thưa** (997 triple/1009 thực thể — trung bình mỗi thực thể chỉ
xuất hiện trong ~2 triple), lại chia 80/20 nên tập test chỉ có ~200 triple
để đánh giá trên toàn bộ không gian 1009 thực thể ứng viên. Với đồ thị thưa
và ít epoch (100) như vậy, TransE khó học được biểu diễn tổng quát hoá tốt —
baseline này dùng để minh hoạ đúng pipeline huấn luyện + đánh giá KGE, không
nhằm đạt SOTA. `PipelineResult` (biến `result`) được Bước 4 dùng lại để
trích embedding.

In [ ]:
if __name__ == "__main__":
    import os
    tf = triples_csv_to_factory("data/raw/triples.csv")
    result = train_transe(tf, num_epochs=100)
    metrics = extract_eval_metrics(result)
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    os.makedirs("reports", exist_ok=True)
    save_eval_json(metrics, "reports/kge_eval.json")

## Bước 4 — Tính độ tương đồng giữa các thực thể

Mỗi thực thể sau khi huấn luyện có một vector embedding. Độ tương đồng
cosine giữa hai vector u, v:

`sim(u, v) = (u · v) / (‖u‖ ‖v‖)`

Giá trị nằm trong [-1, 1]; gần 1 nghĩa là hai thực thể "gần nhau" trong
không gian embedding — với KGE, điều này thường tương ứng với việc hai thực
thể xuất hiện trong ngữ cảnh quan hệ tương tự nhau trên đồ thị (ví dụ hai
nhân vật cùng tham gia một tổ chức, hoặc hai sự kiện cùng một địa điểm).

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


def get_entity_embedding(result, entity_label: str) -> np.ndarray:
    """Lấy vector embedding của 1 thực thể theo label."""
    entity_id = result.training.entities_to_ids([entity_label])[0]
    embedding_tensor = result.model.entity_representations[0](indices=None)
    return embedding_tensor[entity_id].detach().numpy()


def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    """sim(u, v) = (u·v) / (‖u‖‖v‖)."""
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))


def most_similar(result, entity_label: str, candidate_labels: list, top_k: int = 5):
    """Top_k thực thể (trong candidate_labels) có similarity cao nhất với entity_label."""
    target = get_entity_embedding(result, entity_label)
    scored = []
    for label in candidate_labels:
        if label == entity_label:
            continue
        vec = get_entity_embedding(result, label)
        scored.append((label, cosine_similarity(target, vec)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


def plot_embeddings_2d(result, labels: list, out_path: str = "reports/embedding_plot.png") -> None:
    """Giảm chiều bằng PCA và vẽ scatter các thực thể trong labels."""
    vectors = np.array([get_entity_embedding(result, label) for label in labels])
    coords = PCA(n_components=2, random_state=0).fit_transform(vectors)
    plt.figure(figsize=(8, 6))
    plt.scatter(coords[:, 0], coords[:, 1])
    for (x, y), label in zip(coords, labels):
        plt.annotate(label, (x, y))
    plt.title("Thực thể trong không gian embedding (PCA 2D)")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()

### Ví dụ minh hoạ (chạy thật)

Chạy thật `python -m src.step4_similarity`: huấn luyện lại TransE như
Bước 3 (toàn bộ `data/raw/triples.csv` — 997 triple duy nhất / 1009 thực
thể, 100 epoch, `random_seed=42`, cùng model chất lượng thấp với
MRR≈0.0218 đã báo cáo ở Bước 3), lấy 20 thực thể đầu tiên theo
`entity_to_id` để vẽ PCA 2D (lưu tại `reports/embedding_plot.png`), và
tính `most_similar` của thực thể đầu tiên so với toàn bộ 1009 thực thể.

**Cặp có liên hệ trực tiếp trên đồ thị** — `Chiến tranh Việt Nam` và
`Hoa Kỳ` (nối trực tiếp bằng cạnh `P710 người tham gia` — dòng 7 của
`data/raw/triples.csv`):

- `cosine("Chiến tranh Việt Nam", "Hoa Kỳ")` = **-0.1363**

**Cặp không liên quan** — `Chiến tranh Việt Nam` và
`Cúp bóng chuyền nam vô địch châu Á 2012` (hai thực thể không chia sẻ
cạnh nào trong toàn bộ CSV, thuộc hai miền chủ đề khác nhau — chiến
tranh vs. thể thao):

- `cosine("Chiến tranh Việt Nam", "Cúp bóng chuyền nam vô địch châu Á 2012")` = **-0.0359**

**Nhận xét (trung thực, không cherry-pick):** kết quả **không** ủng hộ
giả thuyết "cặp có liên hệ thì similarity cao hơn cặp không liên quan" —
thực tế ngược lại: cặp có cạnh nối trực tiếp (`Chiến tranh Việt Nam` –
`Hoa Kỳ`) cho cosine **thấp hơn** (-0.1363, âm nhiều hơn) so với cặp hoàn
toàn không liên quan (-0.0359). Cả hai giá trị đều âm và gần 0, không có
một hướng lệch dương rõ ràng nào phản ánh cấu trúc đồ thị. Đây là một kết
quả tiêu cực/không thuyết phục, nhưng phù hợp và có thể giải thích được
bằng những gì Bước 3 đã báo cáo: đồ thị quá thưa (997 triple / 1009 thực
thể, trung bình ~2 triple mỗi thực thể — bản thân `Chiến tranh Việt Nam`
chỉ xuất hiện trong 27 dòng, hầu hết cùng một loại quan hệ `P710`), 100
epoch là rất ít, và TransE (baseline) trên một tập test MRR≈0.0218 gần
như ngẫu nhiên (so với random-guess trên 1009 thực thể, MRR ngẫu nhiên kỳ
vọng ≈ 1/1009 ≈ 0.001, nên 0.0218 nhìn thô có "hơn ngẫu nhiên" nhưng vẫn
rất yếu để suy ra semantics đáng tin). Kết luận trung thực: ở baseline
này, cosine similarity giữa embedding **không** phản ánh đáng tin cậy mối
quan hệ thật trên đồ thị — pipeline tính similarity/PCA hoạt động đúng về
mặt kỹ thuật, nhưng chất lượng embedding đầu vào (đã được Bước 3 nêu rõ là
thấp) không đủ để similarity mang ý nghĩa ngữ nghĩa. Scatter PCA ở
`reports/embedding_plot.png` (20 thực thể đầu, phần lớn là các mốc thời
gian dạng `YYYY-01-01T00:00:00Z` cùng hai thực thể `10th`/`11th Vietnam
Film Festival`) cũng không cho thấy cụm chủ đề rõ ràng — cùng một quan
sát. `most_similar` của thực thể đầu tiên
(`-0110-01-01T00:00:00Z`, một mốc thời gian) trả về top-5 gồm cả một sự
kiện lịch sử (`Trận Ngọc Hồi – Đống Đa`, 0.437) lẫn các mốc thời gian
khác — không có mẫu hình rõ ràng nào để kết luận thêm.

In [ ]:
if __name__ == "__main__":
    from src.step3_train_kge import triples_csv_to_factory, train_transe

    tf = triples_csv_to_factory("data/raw/triples.csv")
    result = train_transe(tf, num_epochs=100)
    all_labels = list(tf.entity_to_id.keys())
    sample_labels = all_labels[:20]
    plot_embeddings_2d(result, sample_labels)
    if len(all_labels) >= 2:
        print(most_similar(result, all_labels[0], all_labels, top_k=5))

## Bước 5 — Trích chuỗi quan hệ làm ngữ cảnh

Để sinh câu hỏi có kiểm soát ở Bước 6, mô hình ngôn ngữ không được "tự nhớ"
kiến thức lịch sử — mọi dữ kiện trong câu hỏi phải xuất phát từ một tập fact
cụ thể, kiểm chứng lại được. Bước này lấy tập fact đó bằng cách truy vấn
Cypher multi-hop quanh một thực thể mục tiêu trong Neo4j: đi tối đa
`max_hops` cạnh từ thực thể đó, lấy mọi quan hệ gặp trên đường đi.

In [ ]:
def get_relation_chain(driver, qid: str, max_hops: int = 2, limit: int = 20) -> list:
    """Lấy các quan hệ trong bán kính max_hops quanh thực thể qid."""
    query = f"""
    MATCH path = (e:Entity {{qid: $qid}})-[:RELATION*1..{max_hops}]-(other)
    UNWIND relationships(path) AS rel
    WITH DISTINCT startNode(rel) AS s, rel, endNode(rel) AS o
    RETURN s.label AS subject, rel.label AS relation, o.label AS object,
           rel.pid AS source_pid
    LIMIT $limit
    """
    with driver.session() as session:
        records = session.run(query, qid=qid, limit=limit)
        return [dict(r) for r in records]


def format_context(facts: list) -> str:
    """Chuyển list fact thành đoạn ngữ cảnh có cấu trúc, đánh số."""
    lines = []
    for i, fact in enumerate(facts, start=1):
        lines.append(
            f"{i}. {fact['subject']} --{fact['relation']}--> {fact['object']} "
            f"(nguồn: {fact['source_pid']})"
        )
    return "\n".join(lines)

### Ví dụ minh hoạ (chạy thật)

_(Chọn 1 thực thể quan trọng trong subgraph, in ra chuỗi quan hệ thật quanh
nó, dán nguyên văn `format_context(...)` vào đây làm ví dụ cho báo cáo.)_

Thực thể chọn: **Q8740** — "Chiến tranh Việt Nam" (xuất hiện ở 27 dòng làm
head trong `data/raw/triples.csv`, nên có nhiều quan hệ trong Neo4j). Chạy
`python -m src.step5_relation_chain` trên dữ liệu thật (1017 nút / 1000
cạnh) cho kết quả sau (20 fact đầu, `max_hops=2`, nguyên văn output thật):

```
1. Chiến tranh Việt Nam --người tham gia--> Hoa Kỳ (nguồn: P710)
2. Sự kiện Tết Mậu Thân --người tham gia--> Hoa Kỳ (nguồn: P710)
3. Trận Đồi Thịt Băm --người tham gia--> Hoa Kỳ (nguồn: P710)
4. Trận Long Tân --người tham gia--> Hoa Kỳ (nguồn: P710)
5. Trận Ia Đrăng --người tham gia--> Hoa Kỳ (nguồn: P710)
6. Chiến tranh Đông Dương lần thứ ba --người tham gia--> Hoa Kỳ (nguồn: P710)
7. Chiến dịch Cedar Falls --người tham gia--> Hoa Kỳ (nguồn: P710)
8. Chiến dịch Đắk Tô-Tân Cảnh (1967) --người tham gia--> Hoa Kỳ (nguồn: P710)
9. Trận Nam Đông --người tham gia--> Hoa Kỳ (nguồn: P710)
10. Trận Đồng Xoài --người tham gia--> Hoa Kỳ (nguồn: P710)
11. Chiến dịch Masher --người tham gia--> Hoa Kỳ (nguồn: P710)
12. Vụ tập kích Sơn Tây --người tham gia--> Hoa Kỳ (nguồn: P710)
13. Trận Mậu Thân tại Huế --người tham gia--> Hoa Kỳ (nguồn: P710)
14. Battle of Hill 881 --người tham gia--> Hoa Kỳ (nguồn: P710)
15. Operation Dewey Canyon --người tham gia--> Hoa Kỳ (nguồn: P710)
16. Trận Bình Giã --người tham gia--> Hoa Kỳ (nguồn: P710)
17. Operation Crimp --người tham gia--> Hoa Kỳ (nguồn: P710)
18. Operation Francis Marion --người tham gia--> Hoa Kỳ (nguồn: P710)
19. Chiến dịch Speedy Express --người tham gia--> Hoa Kỳ (nguồn: P710)
20. Trận Tầm Bó --người tham gia--> Hoa Kỳ (nguồn: P710)
```

Vì truy vấn đi hai chiều (`-[:RELATION*1..2]-`, không chỉ theo hướng cạnh),
hop thứ hai từ "Chiến tranh Việt Nam" đi tới "Hoa Kỳ" rồi vòng ra các thực
thể khác cũng có cạnh `P710 (người tham gia)` tới "Hoa Kỳ" — nên 20 fact
đầu (không có `ORDER BY`, giới hạn bởi `LIMIT`) đều xoay quanh các trận
đánh/chiến dịch có Hoa Kỳ tham gia, chứ không lộ ra các quan hệ 1-hop khác
(`vị trí`, `ngày bắt đầu`, ...) của chính Chiến tranh Việt Nam — những quan
hệ đó vẫn có trong Neo4j nhưng bị Cypher trả về sau trong danh sách không
sắp thứ tự này.

In [ ]:
if __name__ == "__main__":
    from src.step2_load_neo4j import get_driver

    driver = get_driver("bolt://localhost:7687", "neo4j", "kg-research-pw")
    # Thay QID dưới đây bằng một QID thật có trong data/raw/triples.csv
    example_qid = "Q8740"
    facts = get_relation_chain(driver, example_qid, max_hops=2, limit=20)
    print(format_context(facts))
    driver.close()

## Bước 6 — Sinh câu hỏi có kiểm soát

Prompt chỉ chứa các fact đã trích ở Bước 5 — mô hình không được dùng kiến
thức nền riêng của nó. Sau khi mô hình trả lời, một bước kiểm tra tự động
đối chiếu lại với chính các fact đầu vào theo 2 tiêu chí:

1. **Năm**: mọi số 4 chữ số xuất hiện trong câu hỏi/đáp án/giải thích phải
   có mặt trong các fact đầu vào (trích bằng regex `\b\d{4}\b`, đối chiếu
   tập hợp).
2. **Anchor**: mô hình phải tự khai những thực thể nó dùng (`anchors`), và
   mọi anchor đó phải là một `subject`/`object` có thật trong fact đầu vào.

Lý do dùng "anchor" thay vì cố dò tên riêng bằng regex trên toàn câu: tiếng
Việt không viết hoa tên riêng theo quy tắc chặt như tiếng Anh, dò bằng regex
dễ sai (bỏ sót hoặc bắt nhầm từ thường). Bắt mô hình khai báo anchor rồi đối
chiếu tập hợp là một kiểm tra đáng tin cậy hơn nhiều.

In [ ]:
import re
import json


def build_prompt(context: str) -> str:
    return (
        "Dưới đây là các dữ kiện lịch sử đã được xác minh:\n\n"
        f"{context}\n\n"
        "Hãy soạn một câu hỏi kiểm tra kiến thức dựa CHỈ trên các dữ kiện trên. "
        "Không dùng bất kỳ tên riêng hay năm nào không xuất hiện trong dữ kiện. "
        "Trả về đúng định dạng JSON, không thêm chữ nào khác:\n"
        '{"question": "...", "answer": "...", "explanation": "...", '
        '"anchors": ["..."]}'
    )


def call_llm(prompt: str, client=None) -> str:
    """Gọi Claude API, trả về text thô của phản hồi.
    `client` truyền vào để test bằng fake client, không gọi mạng thật."""
    if client is None:
        import anthropic
        client = anthropic.Anthropic()
    message = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text


def extract_years(text: str) -> set:
    """Trích mọi số 4 chữ số trong text."""
    return set(re.findall(r"\b\d{4}\b", text))


def extract_allowed_years(facts: list) -> set:
    """Trích mọi năm 4 chữ số xuất hiện trong facts đầu vào."""
    allowed = set()
    for fact in facts:
        allowed |= extract_years(fact["subject"]) | extract_years(fact["object"])
    return allowed


def extract_allowed_names(facts: list) -> set:
    """Trích mọi subject/object xuất hiện trong facts đầu vào."""
    names = set()
    for fact in facts:
        names.add(fact["subject"])
        names.add(fact["object"])
    return names


def validate_question(question_json: dict, facts: list) -> dict:
    """Kiểm tra câu hỏi: (1) mọi năm phải có trong ngữ cảnh, (2) mọi anchor
    mô hình khai phải là một thực thể có thật trong facts đầu vào."""
    full_text = " ".join([
        question_json.get("question", ""),
        question_json.get("answer", ""),
        question_json.get("explanation", ""),
    ])
    used_years = extract_years(full_text)
    allowed_years = extract_allowed_years(facts)
    unmatched_years = used_years - allowed_years
    if unmatched_years:
        return {"accepted": False,
                "reason": f"Năm không có trong ngữ cảnh: {unmatched_years}"}

    allowed_names = extract_allowed_names(facts)
    anchors = set(question_json.get("anchors", []))
    if not anchors:
        return {"accepted": False, "reason": "Không có anchor nào được khai báo"}
    unmatched_anchors = anchors - allowed_names
    if unmatched_anchors:
        return {"accepted": False,
                "reason": f"Anchor không có trong ngữ cảnh: {unmatched_anchors}"}

    return {"accepted": True, "reason": None}

### Ví dụ minh hoạ (chạy thật)

_(Chưa chạy thật — cần biến môi trường `ANTHROPIC_API_KEY`. Sau khi có key, chạy `python -m src.step6_generate_question` với ngữ cảnh đầy đủ để lấy ví dụ "accepted", rồi rút bớt 1 fact và chạy lại để lấy ví dụ "rejected"; dán cả hai kết quả thật vào đây.)_

In [ ]:
if __name__ == "__main__":
    from src.step2_load_neo4j import get_driver
    from src.step5_relation_chain import get_relation_chain, format_context

    driver = get_driver("bolt://localhost:7687", "neo4j", "kg-research-pw")
    example_qid = "Q10520"  # thay bằng QID thật
    facts = get_relation_chain(driver, example_qid, max_hops=2, limit=20)
    driver.close()

    context = format_context(facts)
    prompt = build_prompt(context)
    raw_response = call_llm(prompt)
    question_json = json.loads(raw_response)
    result = validate_question(question_json, facts)
    print(json.dumps(question_json, ensure_ascii=False, indent=2))
    print(result)

## Kết luận và hướng tiếp theo

Pipeline đã chứng minh được trên dữ liệu Wikidata công khai: trích subgraph
→ lưu Neo4j → huấn luyện KGE → tính độ tương đồng → trích chuỗi quan hệ →
sinh câu hỏi có kiểm soát với validate tự động chặn được nội dung sai lệch.

Bước tiếp theo (ngoài phạm vi báo cáo này): áp dụng đúng pipeline này lên dữ
liệu lịch sử Việt Nam tự soạn giai đoạn 1858–1945, rồi triển khai tầng lập
lịch (Adaptive Microlearning Scheduling), tầng tương tác và tầng cập nhật
đánh giá (Relational Diffusion) theo 5 tầng giáo viên đã giao.